# Notebook Overview — Generate Autoencoder Video Representations

## Purpose

This notebook loads and validates the standardized autoencoder representations generated by Notebook 03 for downstream representation-based VideoQA experiments.

Rather than training the autoencoder or generating new latent embeddings, the notebook restores the trained model and the existing segment-level and video-level representation files, standardizes the video representation schema, verifies the train, validation, and test representation records, and generates a representation summary report.

The resulting `autoencoder_video` representation artifact is consumed by Notebook 07, which selects the required training and validation representations and merges them with the corresponding NExT-QA annotations and shared CLIP text embeddings.

## Inputs

* Trained autoencoder model generated by Notebook 03
* Autoencoder segment-level latent representations generated by Notebook 03
* Autoencoder video-level latent representations generated by Notebook 03
* NExT-QA annotations
* Shared project configuration

## Outputs

* Standardized autoencoder video representation file
* Validated segment-level and video-level representation data
* Autoencoder representation summary report
* Notebook 07-compatible representation artifacts

## Processing Workflow

1. Initialize the project environment.
2. Configure the autoencoder representation-validation workflow.
3. Verify the runtime environment.
4. Load the trained autoencoder model.
5. Prepare a development evaluation dataset for configuration and annotation validation.
6. Load and standardize the existing segment-level and video-level autoencoder representations.
7. Validate the standardized representation schema and embedding values.
8. Verify the loaded representation artifacts.
9. Generate and save the autoencoder representation summary report.
10. Display representative video representation records.
11. Summarize the completed representation-validation workflow.

## Downstream Consumer

Notebook 07 — Run Representation-Based VideoQA


### 🔷 Step 1 — Initialize Environment for Autoencoder Representation Preparation

* Initialize the notebook runtime and prepare the project execution environment.
* Mount Google Drive and restore required project resources.
* Clone the project repository and load shared configuration constants and utility modules.
* Verify that the required datasets and experiment directories are available.
* Confirm readiness for autoencoder representation preparation.



In [ ]:
# ============================================================
# Step 1: Initialize Environment for Autoencoder Representation Extraction
# ============================================================

VERBOSE = True
REQUIRE_L4_GPU = True
EXPECTED_NEXTQA_VIDEO_COUNT = 5440

import os
import shutil
import time
from pathlib import Path

import pandas as pd

from google.colab import drive, userdata

print("Initializing notebook environment...")
print("-" * 60)

# ------------------------------------------------------------
# DRIVE MOUNT (REQUIRED)
# ------------------------------------------------------------

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):
    print("Mounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)
else:
    print("Google Drive already mounted.")

# ------------------------------------------------------------
# REPO CONFIG
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

os.chdir(REPO_BASE_DIR)

# ------------------------------------------------------------
# CLONE REPOSITORY
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = (
    f"https://{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

os.chdir(REPO_BASE_DIR)

if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# LOAD PROJECT MODULES
# ------------------------------------------------------------

print("\nLoading project configuration and utility modules...")

from src.videoqa_representation_config import *

# ------------------------------------------------------------
# NOTEBOOK-SPECIFIC EXPERIMENT SELECTION
# ------------------------------------------------------------

#EXPERIMENT_NAME = "ae_seg6s_stride4_dev25"
EXPERIMENT_NAME = "ae_seg6s_stride4_dev100"

configure_experiment(EXPERIMENT_NAME)


# NOTE: KEEP (needed for evaluation)
from src.training_validation import *
from src.autoencoder_model import *

# ------------------------------------------------------------
# OPTIONAL DATA MODULES (Notebook 04 SAFE SET)
# ------------------------------------------------------------

from src.nextqa_metadata import *

# ------------------------------------------------------------
# OUTPUT SETUP
# ------------------------------------------------------------

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [
    Path("src"),
    Path("datasets"),
    OUTPUTS_DIR,
]

missing_paths = [p for p in required_paths if not p.exists()]

if missing_paths:
    for p in missing_paths:
        print(f"Missing required path: {p}")

    raise FileNotFoundError("One or more required project paths are missing.")

print("Configuration loaded.")
print("Project paths initialized.")

# ------------------------------------------------------------
# LOAD ONLY ANNOTATIONS
# ------------------------------------------------------------

print("\nLoading NExT-QA annotations (evaluation labels only)...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

print("\nDataset metadata loaded.")
print(f"Annotation records: {len(annotations_df):,}")

# ------------------------------------------------------------
# VERIFY AUTOENCODER MODEL
# ------------------------------------------------------------

AUTOENCODER_MODEL_PATH = (
    Path("/content/drive/MyDrive/VideoQA_Project/experiments")
    / EXPERIMENT_NAME
    / "autoencoder"
    / "models"
    / "autoencoder.pt"
)

print("\nChecking Notebook 03 autoencoder model...")

if not AUTOENCODER_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing trained autoencoder model: {AUTOENCODER_MODEL_PATH}"
    )

print("Notebook 03 autoencoder model found.")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nEnvironment initialization complete.")
print("-" * 60)
print("Notebook ready for embedding extraction.")



### 🔷 Step 2 — Define Autoencoder Representation Configuration

* Define the configuration used to load and validate the autoencoder representation artifacts.
* Use the active experiment configuration established during environment initialization.
* Configure the evaluation split, development subset size, and randomization settings.
* Record the autoencoder frame, segment-sampling, and batch-size settings associated with the experiment.
* Define inference-only operation for the representation workflow.
* Display and validate the active representation configuration.




In [ ]:
# ============================================================
# Step 2: Define Autoencoder Representation Configuration
# ============================================================

print("\nDefining autoencoder representation configuration...\n")

# ------------------------------------------------------------
# Import shared constants (single source of truth)
# ------------------------------------------------------------
from src.videoqa_representation_config import *

# ------------------------------------------------------------
# Representation Extraction Configuration
# ------------------------------------------------------------

REPRESENTATION_CONFIG = {
    # Dataset control
    "evaluation_split": EVALUATION_SPLIT,
    "development_subset_size": DEVELOPMENT_SUBSET_SIZE,
    "random_seed": RANDOM_SEED,

    # Autoencoder representation settings
    "frame_size": AUTOENCODER_FRAME_SIZE,
    "frames_per_segment": AUTOENCODER_FRAMES_PER_SEGMENT,
    "batch_size": AUTOENCODER_BATCH_SIZE,

    # Model usage mode (STRICTLY INFERENCE ONLY)
    "inference_mode": "autoencoder_latent_extraction",

    # Output control
    "save_latent_vectors": True,
    "verbose": True,
}

# ------------------------------------------------------------
# Validate Configuration
# ------------------------------------------------------------

assert REPRESENTATION_CONFIG["development_subset_size"] > 0
assert REPRESENTATION_CONFIG["frame_size"] > 0
assert REPRESENTATION_CONFIG["frames_per_segment"] > 0
assert REPRESENTATION_CONFIG["batch_size"] > 0

# ------------------------------------------------------------
# Display Configuration
# ------------------------------------------------------------

print("Representation Extraction Configuration:")
for key, value in REPRESENTATION_CONFIG.items():
    print(f"  {key:<30}: {value}")



### 🔷 Step 3 — Verify Runtime Environment and Dependencies

* Display Python, platform, PyTorch, and CUDA runtime information.
* Verify GPU availability and enforce the configured L4 GPU requirement when enabled.
* Display detected GPU hardware and memory information.
* Verify required Python package availability.
* Confirm runtime readiness before loading the trained autoencoder and representation files.


In [ ]:
# ============================================================
# Step 3: Verify Runtime Environment and Dependencies
# ============================================================

import sys
import platform
import importlib

print("Verifying runtime environment and dependencies...\n")

# ------------------------------------------------------------
# Runtime Information
# ------------------------------------------------------------

print("Runtime Information")
print("-" * 60)
print(f"Python Version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")

# ------------------------------------------------------------
# PyTorch / CUDA Verification
# ------------------------------------------------------------

device = "cpu"

try:
    import torch

    print("\nPyTorch Information")
    print("-" * 60)
    print(f"PyTorch Version : {torch.__version__}")
    print(f"CUDA Available  : {torch.cuda.is_available()}")

    if torch.cuda.is_available():

        print(f"CUDA Version    : {torch.version.cuda}")
        print(f"GPU Count       : {torch.cuda.device_count()}")

        for idx in range(torch.cuda.device_count()):

            gpu_name = torch.cuda.get_device_name(idx)
            gpu_props = torch.cuda.get_device_properties(idx)

            total_memory_gb = gpu_props.total_memory / (1024 ** 3)

            print(f"GPU {idx}          : {gpu_name}")
            print(f"GPU {idx} Memory   : {total_memory_gb:.1f} GB")

        allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
        reserved_gb = torch.cuda.memory_reserved() / (1024 ** 3)

        print(f"Allocated Memory : {allocated_gb:.2f} GB")
        print(f"Reserved Memory  : {reserved_gb:.2f} GB")

        primary_gpu = torch.cuda.get_device_name(0)

        if REQUIRE_L4_GPU and "L4" not in primary_gpu:
            raise RuntimeError(
                f"Required NVIDIA L4 GPU not available. "
                f"Detected GPU: {primary_gpu}."
            )

        device = "cuda"

    else:
        print("WARNING: No CUDA GPU detected.")
        device = "cpu"

except Exception as e:
    print(f"ERROR: PyTorch not available ({e})")
    device = "cpu"

# ------------------------------------------------------------
# Dependency Verification (Runtime Environment Gate)
# ------------------------------------------------------------

required_packages = [
    "torch",
    "transformers",
    "accelerate",
    "numpy",
    "pandas",
]

print("\nDependency Verification")
print("-" * 60)

dependency_status = []

for package_name in required_packages:

    try:
        module = importlib.import_module(package_name)
        version = getattr(module, "__version__", "unknown")

        dependency_status.append({
            "package": package_name,
            "status": "OK",
            "version": version,
        })

        print(f"[OK]   {package_name:<15} {version}")

    except Exception:

        dependency_status.append({
            "package": package_name,
            "status": "MISSING",
            "version": "",
        })

        print(f"[FAIL] {package_name}")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

missing_packages = [
    item["package"]
    for item in dependency_status
    if item["status"] != "OK"
]

print("\nVerification Summary")
print("-" * 60)
print(f"Execution Device : {device}")

if len(missing_packages) == 0:
    print("All required dependencies are available.")
else:
    print("Missing packages:")
    for pkg in missing_packages:
        print(f"  - {pkg}")



### 🔷 Step 4 — Load Trained Autoencoder Model

* Locate the trained autoencoder model artifacts for the active experiment.
* Verify that the trained model artifacts exist in the Google Drive experiment directory.
* Import the shared autoencoder model definition from the project source tree.
* Restore the trained model weights from the saved trained model artifacts.
* Move the model to the active computation device.
* Set the autoencoder to evaluation mode for representation workflows.

In [ ]:
# ============================================================
# Step 4: Load Trained Autoencoder Model
# ============================================================

import torch

print("Loading trained autoencoder model...")

# ------------------------------------------------------------
# Checkpoint path from experiment configuration
# ------------------------------------------------------------

CHECKPOINT_PATH = AUTOENCODER_MODEL_PATH

print(f"Checkpoint path:\n{CHECKPOINT_PATH}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Autoencoder checkpoint not found at expected experiment path:\n"
        f"{CHECKPOINT_PATH}"
    )

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

from src.autoencoder_model import ConvAutoencoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ConvAutoencoder().to(device)

# ------------------------------------------------------------
# Load weights
# ------------------------------------------------------------

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

# ------------------------------------------------------------
# Confirm
# ------------------------------------------------------------

print("Autoencoder model loaded successfully.")
print(f"Device: {device}")



### 🔷 Step 5 — Prepare Development Evaluation Dataset

* Select the configured evaluation split from the NExT-QA annotations.
* Sample the configured development subset using the project random seed.
* Validate required video, question, answer, and answer-choice columns.
* Attach the ground-truth answer text for each evaluation record.
* Add representation metadata fields for the active experiment.
* Validate the prepared evaluation dataset before representation alignment.

In [ ]:
# ============================================================
# Step 5: Prepare Development Evaluation Dataset
# ============================================================

from pathlib import Path
import pandas as pd

print("Preparing autoencoder development evaluation subset...")

evaluation_split = EVALUATION_SPLIT
development_subset_size = DEVELOPMENT_SUBSET_SIZE
random_seed = RANDOM_SEED

# ------------------------------------------------------------
# Validate annotation columns
# ------------------------------------------------------------

required_annotation_columns = [
    "split",
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    GROUND_TRUTH_ANSWER_COLUMN,
] + CHOICE_COLUMNS

missing_annotation_columns = [
    col for col in required_annotation_columns
    if col not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        f"annotations_df is missing required columns: {missing_annotation_columns}"
    )

# ------------------------------------------------------------
# Select evaluation split and development subset
# ------------------------------------------------------------

eval_df = annotations_df[
    annotations_df["split"] == evaluation_split
].copy()

if len(eval_df) == 0:
    raise ValueError(f"No records found for split: {evaluation_split}")

sample_size = min(
    development_subset_size,
    len(eval_df),
)

eval_df = (
    eval_df
    .sample(
        n=sample_size,
        random_state=random_seed,
    )
    .reset_index(drop=True)
)

print(f"Selected evaluation samples: {len(eval_df):,}")

# ------------------------------------------------------------
# Attach ground-truth answer text
# ------------------------------------------------------------

def answer_index_to_text(row):
    answer_idx = int(row[GROUND_TRUTH_ANSWER_COLUMN])
    option_col = f"a{answer_idx}"

    if option_col not in CHOICE_COLUMNS:
        raise ValueError(f"Answer option column not valid: {option_col}")

    return row[option_col]

eval_df["ground_truth_text"] = eval_df.apply(
    answer_index_to_text,
    axis=1,
)

# ------------------------------------------------------------
# Add representation source metadata
# ------------------------------------------------------------

eval_df["video_source"] = "autoencoder_latent"
eval_df["representation_experiment"] = EXPERIMENT_NAME

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

required_eval_columns = [
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    GROUND_TRUTH_ANSWER_COLUMN,
    *CHOICE_COLUMNS,
    "ground_truth_text",
    "video_source",
    "representation_experiment",
]

missing_columns = [
    col for col in required_eval_columns
    if col not in eval_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required evaluation columns: {missing_columns}"
    )

print("\nAutoencoder evaluation dataset prepared successfully.")
print(f"Evaluation samples : {len(eval_df):,}")
print(f"Evaluation split   : {evaluation_split}")
print(f"Random seed        : {random_seed}")
print(f"Answer mode        : {ANSWER_MODE}")
print(f"Video source       : autoencoder_latent")

print("\nEvaluation Dataset Preview:")
display(eval_df.head())



### 🔷 Step 6 — Load and Standardize Autoencoder Video Representations

* Locate the segment-level and video-level autoencoder representation files generated by Notebook 03.
* Verify that both representation artifacts exist before loading them.
* Load the segment-level and video-level representation datasets.
* Validate the required video identifiers, split information, segment counts, representation metadata, and embedding columns.
* Standardize video identifiers and representation metadata using the shared Notebook 07-compatible schema.
* Confirm that the expected training, validation, and test split values are valid and that the configured evaluation split is available.
* Check for duplicate identifiers and missing embedding values.
* Save the standardized video representation artifact and create compatibility aliases for downstream processing.



In [ ]:
# ============================================================
# Step 6: Load and Standardize Autoencoder Video Representations
# ============================================================

import pandas as pd
import re

print("Loading and standardizing autoencoder latent representations...")

# ------------------------------------------------------------
# Locate Drive-based representation outputs
# ------------------------------------------------------------

segment_representations_path = AUTOENCODER_SEGMENT_REPRESENTATIONS_CSV
video_representations_path = AUTOENCODER_VIDEO_REPRESENTATIONS_CSV

print(f"Segment representation path:\n{segment_representations_path}")
print(f"\nVideo representation path:\n{video_representations_path}")

if not segment_representations_path.exists():
    raise FileNotFoundError(
        f"Segment representations not found: {segment_representations_path}"
    )

if not video_representations_path.exists():
    raise FileNotFoundError(
        f"Video representations not found: {video_representations_path}"
    )

ae_segment_representation_df = pd.read_csv(segment_representations_path)
ae_video_representation_df = pd.read_csv(video_representations_path)

# ------------------------------------------------------------
# Validate required video representation columns
# ------------------------------------------------------------

required_video_columns = [
    "video",
    "split",
    "segment_count",
    "representation_experiment",
    "representation_source",
]

missing_video_columns = [
    col for col in required_video_columns
    if col not in ae_video_representation_df.columns
]

if missing_video_columns:
    raise ValueError(
        f"ae_video_representation_df is missing columns: {missing_video_columns}"
    )

video_embedding_columns = [
    col for col in ae_video_representation_df.columns
    if re.fullmatch(r"embedding_\d{3}", col)
]

if len(video_embedding_columns) == 0:
    raise ValueError("No standardized embedding columns found.")

video_embedding_columns = sorted(video_embedding_columns)

if len(video_embedding_columns) != AUTOENCODER_LATENT_DIM:
    raise ValueError(
        f"Expected {AUTOENCODER_LATENT_DIM} embedding dimensions, "
        f"found {len(video_embedding_columns)}."
    )

# ------------------------------------------------------------
# Standardize identifiers and metadata
# ------------------------------------------------------------

ae_video_representation_df["video"] = (
    ae_video_representation_df["video"].astype(str)
)

ae_video_representation_df["split"] = (
    ae_video_representation_df["split"].astype(str)
)

ae_video_representation_df["record_id"] = (
    "ae_video_" + ae_video_representation_df["video"]
)

ae_video_representation_df["representation_source"] = "autoencoder_video"
ae_video_representation_df["representation_type"] = "video"
ae_video_representation_df["model_name"] = "conv_autoencoder"
ae_video_representation_df["embedding_dimension"] = AUTOENCODER_LATENT_DIM

# ------------------------------------------------------------
# Validate split values
# ------------------------------------------------------------

valid_splits = {
    "train",
    "val",
    "test",
}

observed_splits = set(
    ae_video_representation_df["split"]
    .dropna()
    .astype(str)
    .unique()
)

invalid_splits = sorted(observed_splits - valid_splits)

if invalid_splits:
    raise ValueError(
        f"Unexpected split values found in AE video representations: {invalid_splits}"
    )

if EVALUATION_SPLIT not in observed_splits:
    raise ValueError(
        f"Evaluation split '{EVALUATION_SPLIT}' was not found in AE video representations. "
        "Rerun Notebook 03 Step 12 to regenerate train/val/test AE representations."
    )

# ------------------------------------------------------------
# Reorder columns to match shared video representation structure
# ------------------------------------------------------------

metadata_columns = [
    "record_id",
    "video",
    "split",
    "representation_source",
    "representation_type",
    "representation_experiment",
    "model_name",
    "embedding_dimension",
    "segment_count",
]

ae_video_representation_df = ae_video_representation_df[
    metadata_columns + video_embedding_columns
]

# ------------------------------------------------------------
# Validate standardized schema
# ------------------------------------------------------------

duplicate_record_ids = ae_video_representation_df["record_id"].duplicated().sum()

if duplicate_record_ids > 0:
    raise ValueError(
        f"Found {duplicate_record_ids} duplicate autoencoder record_id values."
    )

duplicate_video_ids = ae_video_representation_df["video"].duplicated().sum()

if duplicate_video_ids > 0:
    raise ValueError(
        f"Found {duplicate_video_ids} duplicate autoencoder video values."
    )

missing_embedding_values = (
    ae_video_representation_df[video_embedding_columns]
    .isna()
    .sum()
    .sum()
)

if missing_embedding_values > 0:
    raise ValueError(
        f"Found {missing_embedding_values} missing embedding values."
    )

# ------------------------------------------------------------
# Save standardized video representation artifact
# ------------------------------------------------------------

ae_video_representation_df.to_csv(
    video_representations_path,
    index=False,
)

# ------------------------------------------------------------
# Notebook 07 compatibility aliases
# ------------------------------------------------------------

standardized_video_representation_df = ae_video_representation_df.copy()
filtered_video_representation_df = ae_video_representation_df.copy()

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\nAutoencoder latent representations loaded and standardized successfully.")
print(f"Segment representations : {len(ae_segment_representation_df):,}")
print(f"Video representations   : {len(ae_video_representation_df):,}")
print(f"Embedding dimensions    : {len(video_embedding_columns):,}")
print(f"Representation source   : autoencoder_video")
print(f"Experiment              : {EXPERIMENT_NAME}")

print("\nVideo Representation Split Counts")
print("-" * 60)

display(
    ae_video_representation_df
    .groupby("split")
    .size()
    .rename("video_count")
    .reset_index()
)

print("\nStandardized Video Representation Preview:")
display(
    ae_video_representation_df[
        [
            "record_id",
            "video",
            "split",
            "representation_source",
            "representation_type",
            "representation_experiment",
            "model_name",
            "embedding_dimension",
            "segment_count",
            *video_embedding_columns[:5],
        ]
    ].head()
)



### 🔷 Step 7 — Validate Autoencoder Representation Dataset

* Verify that the segment-level and standardized video-level representation datasets were loaded successfully.
* Confirm that the video representation dataset follows the shared Notebook 07-compatible schema.
* Validate representation identifiers, split information, source metadata, embedding dimensions, segment counts, and latent feature columns.
* Check for missing or nonnumeric embedding values.
* Detect zero segment counts and duplicate video records.
* Generate and display a validation summary.
* Confirm that the autoencoder video representations are ready for selection and merging in Notebook 07.


In [ ]:
# ============================================================
# Step 7: Validate Autoencoder Representation Dataset
# ============================================================

import pandas as pd

print("Validating autoencoder representation dataset...")

# ------------------------------------------------------------
# Verify required objects from Step 6
# ------------------------------------------------------------

required_objects = [
    "ae_video_representation_df",
    "ae_segment_representation_df",
    "video_embedding_columns",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required objects from Step 6: {missing_objects}"
    )

# ------------------------------------------------------------
# Validate Notebook 07 / CLIP-compatible schema
# ------------------------------------------------------------

required_video_columns = [
    "record_id",
    VIDEO_ID_COLUMN,
    "split",
    "representation_source",
    "representation_type",
    "representation_experiment",
    "model_name",
    "embedding_dimension",
    "segment_count",
    *video_embedding_columns,
]

missing_video_columns = [
    col for col in required_video_columns
    if col not in ae_video_representation_df.columns
]

if missing_video_columns:
    raise ValueError(
        f"ae_video_representation_df is missing columns required by "
        f"Notebook 07 representation schema: {missing_video_columns}"
    )

expected_representation_source = "autoencoder_video"

representation_sources = (
    ae_video_representation_df["representation_source"]
    .dropna()
    .unique()
    .tolist()
)

if representation_sources != [expected_representation_source]:
    raise ValueError(
        f"Expected representation_source='{expected_representation_source}', "
        f"found {representation_sources}."
    )

# ------------------------------------------------------------
# Validate embedding values
# ------------------------------------------------------------

missing_embedding_values = (
    ae_video_representation_df[video_embedding_columns]
    .isna()
    .sum()
    .sum()
)

non_numeric_embedding_columns = [
    col for col in video_embedding_columns
    if not pd.api.types.is_numeric_dtype(ae_video_representation_df[col])
]

zero_segment_count = (
    ae_video_representation_df["segment_count"] <= 0
).sum()

duplicate_video_count = (
    ae_video_representation_df[VIDEO_ID_COLUMN]
    .duplicated()
    .sum()
)

embedding_dimension_values = (
    ae_video_representation_df["embedding_dimension"]
    .dropna()
    .unique()
    .tolist()
)

# ------------------------------------------------------------
# Build validation summary
# ------------------------------------------------------------

validation_summary = {
    "video_representations_loaded": len(ae_video_representation_df),
    "unique_videos": ae_video_representation_df[VIDEO_ID_COLUMN].nunique(),
    "segment_representations_loaded": len(ae_segment_representation_df),
    "embedding_dimensions": len(video_embedding_columns),
    "schema_compatible_with_notebook_07": True,
    "representation_source": expected_representation_source,
    "missing_embedding_values": int(missing_embedding_values),
    "non_numeric_embedding_columns": len(non_numeric_embedding_columns),
    "zero_segment_count": int(zero_segment_count),
    "duplicate_video_records": int(duplicate_video_count),
    "answer_mode": ANSWER_MODE,
    "representation_experiment": EXPERIMENT_NAME,
}

validation_df = pd.DataFrame(
    validation_summary.items(),
    columns=["Validation Check", "Value"],
)

display(validation_df)

# ------------------------------------------------------------
# Raise errors for failed validation checks
# ------------------------------------------------------------

if len(video_embedding_columns) != AUTOENCODER_LATENT_DIM:
    raise ValueError(
        f"Expected {AUTOENCODER_LATENT_DIM} embedding dimensions, "
        f"found {len(video_embedding_columns)}."
    )

if embedding_dimension_values != [AUTOENCODER_LATENT_DIM]:
    raise ValueError(
        f"Expected embedding_dimension={AUTOENCODER_LATENT_DIM}, "
        f"found {embedding_dimension_values}."
    )

if missing_embedding_values > 0:
    raise ValueError(
        f"Found {missing_embedding_values} missing embedding values."
    )

if non_numeric_embedding_columns:
    raise ValueError(
        f"Non-numeric embedding columns found: {non_numeric_embedding_columns[:10]}"
    )

if zero_segment_count > 0:
    raise ValueError(
        f"Found {zero_segment_count} records with zero segment count."
    )

if duplicate_video_count > 0:
    raise ValueError(
        f"Found {duplicate_video_count} duplicate video records."
    )

print(
    "\nRepresentation validation passed. "
    "Autoencoder video representations follow the Notebook 07 / CLIP-compatible schema."
)



### 🔷 Step 8 — Verify Representation Artifacts

* Verify that the segment-level and video-level autoencoder representation objects are available.
* Confirm the loaded segment and video representation counts.
* Report the standardized embedding dimensionality and representation source.
* Confirm that the video representation artifact follows the shared `embedding_###` column convention.
* Verify readiness for Notebook 07, where the representations will be merged with annotations and shared CLIP text embeddings.



In [ ]:
# ============================================================
# Step 8: Verify Representation Artifacts
# ============================================================

print("Verifying representation artifacts...")

required_objects = [
    "ae_video_representation_df",
    "ae_segment_representation_df",
    "video_embedding_columns",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing representation artifacts: {missing_objects}"
    )

print("Representation artifacts verified.")

print(f"Video representations   : {len(ae_video_representation_df):,}")
print(f"Segment representations : {len(ae_segment_representation_df):,}")
print(f"Embedding dimensions    : {len(video_embedding_columns):,}")

print("\nNotebook 07 will merge these representations with")
print("the evaluation dataset and shared CLIP text embeddings.")

print("\nShared representation schema:")
print(f"Representation source   : autoencoder_video")
print(f"Embedding columns       : {video_embedding_columns[0]} ... {video_embedding_columns[-1]}")



### 🔷 Step 9 — Generate Autoencoder Representation Summary Report

* Generate summary statistics for the loaded autoencoder representation artifacts.
* Report the experiment name, representation source and type, model name, and answer mode.
* Summarize segment-level and video-level representation counts and the number of unique videos.
* Record the observed and expected embedding dimensionality.
* Calculate minimum, maximum, and mean segment counts across represented videos.
* Report the number of missing embedding values.
* Save the summary report to the active Google Drive autoencoder representation directory.


In [ ]:
# ============================================================
# Step 9: Generate Autoencoder Representation Summary Report
# ============================================================

import pandas as pd

print("Generating autoencoder representation summary report...")

required_objects = [
    "ae_video_representation_df",
    "ae_segment_representation_df",
    "video_embedding_columns",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required objects from Step 6: {missing_objects}"
    )

# ------------------------------------------------------------
# Build summary rows
# ------------------------------------------------------------

summary_rows = [
    {"metric": "experiment_name", "value": EXPERIMENT_NAME},
    {"metric": "experiment_type", "value": "autoencoder_representation"},
    {"metric": "representation_source", "value": "autoencoder_video"},
    {"metric": "representation_type", "value": "video"},
    {"metric": "model_name", "value": "conv_autoencoder"},
    {"metric": "answer_mode", "value": ANSWER_MODE},
    {"metric": "video_representations", "value": len(ae_video_representation_df)},
    {"metric": "unique_videos", "value": ae_video_representation_df[VIDEO_ID_COLUMN].nunique()},
    {"metric": "segment_representations", "value": len(ae_segment_representation_df)},
    {"metric": "embedding_dimensions", "value": len(video_embedding_columns)},
    {"metric": "expected_embedding_dimensions", "value": AUTOENCODER_LATENT_DIM},
    {"metric": "minimum_segment_count", "value": int(ae_video_representation_df["segment_count"].min())},
    {"metric": "maximum_segment_count", "value": int(ae_video_representation_df["segment_count"].max())},
    {"metric": "mean_segment_count", "value": round(ae_video_representation_df["segment_count"].mean(), 2)},
    {
        "metric": "missing_embedding_values",
        "value": int(
            ae_video_representation_df[video_embedding_columns]
            .isna()
            .sum()
            .sum()
        ),
    },
]

autoencoder_summary_df = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# Save summary report
# ------------------------------------------------------------

AUTOENCODER_REPRESENTATIONS_DRIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

autoencoder_summary_csv = (
    AUTOENCODER_REPRESENTATIONS_DRIVE_DIR
    / "autoencoder_representation_summary.csv"
)

autoencoder_summary_df.to_csv(
    autoencoder_summary_csv,
    index=False,
)

print("Autoencoder representation summary report saved.")
print(f"Summary file : {autoencoder_summary_csv}")

display(autoencoder_summary_df)



### 🔷 Step 10 — Display Sample Representation Records

* Randomly sample standardized autoencoder video representation records for inspection.
* Display video identifiers, segment counts, representation metadata, and the first five embedding dimensions.
* Report the total number of video representations and unique videos.
* Summarize the embedding dimensionality and segment-count statistics.
* Support qualitative verification of the standardized video representation artifact.


In [ ]:
# ============================================================
# Step 10: Display Sample Representation Records
# ============================================================

import pandas as pd

required_objects = [
    "ae_video_representation_df",
    "video_embedding_columns",
]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing required objects from Step 6: {missing_objects}"
    )

sample_count = min(
    10,
    len(ae_video_representation_df),
)

sample_representation_df = (
    ae_video_representation_df
    .sample(
        n=sample_count,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

display_columns = [
    VIDEO_ID_COLUMN,
    "segment_count",
    "representation_source",
    "representation_type",
    "representation_experiment",
    *video_embedding_columns[:5],
]

print(f"Displaying {sample_count} autoencoder video representations...")
print("Representation source : autoencoder_video")
print(f"Embedding dimensions  : {len(video_embedding_columns)}")
print("\nShowing first 5 embedding dimensions only.\n")

display(
    sample_representation_df[
        display_columns
    ]
)

# ------------------------------------------------------------
# Representation Summary
# ------------------------------------------------------------

print("\nRepresentation Summary")
print("-" * 60)
print(f"Video representations   : {len(ae_video_representation_df):,}")
print(f"Unique videos           : {ae_video_representation_df[VIDEO_ID_COLUMN].nunique():,}")
print(f"Embedding dimensions    : {len(video_embedding_columns):,}")
print(f"Min segment count       : {int(ae_video_representation_df['segment_count'].min())}")
print(f"Max segment count       : {int(ae_video_representation_df['segment_count'].max())}")
print(f"Mean segment count      : {ae_video_representation_df['segment_count'].mean():.2f}")



### 🔷 Step 11 — Notebook Summary

* Summarize the completed autoencoder representation loading, standardization, and validation workflow.
* Report representation counts for the training, validation, and test splits.
* Display segment-level and video-level representation statistics and embedding dimensionality.
* Identify the standardized representation files and generated summary report.
* Confirm compatibility with the shared Notebook 07 representation schema.
* Describe how Notebook 07 will combine the autoencoder video representations with NExT-QA annotations and shared CLIP text embeddings.



In [ ]:
# ============================================================
# Step 11: Notebook Summary
# ============================================================

print("Notebook 04 complete.")
print("=" * 60)

split_counts = (
    ae_video_representation_df
    .groupby("split")
    .size()
    .to_dict()
)

print("\nAutoencoder Representation Experiment")
print("-" * 60)
print(f"Experiment name          : {EXPERIMENT_NAME}")
print(f"Training split           : train")
print(f"Evaluation split         : {EVALUATION_SPLIT}")
print(f"Development subset size  : {DEVELOPMENT_SUBSET_SIZE}")
print(f"Answer mode              : {ANSWER_MODE}")

print("\nRepresentation Dataset")
print("-" * 60)
print(f"Video representations    : {len(ae_video_representation_df):,}")
print(f"  Training videos        : {split_counts.get('train', 0):,}")
print(f"  Validation videos      : {split_counts.get('val', 0):,}")
print(f"  Test videos            : {split_counts.get('test', 0):,}")
print(f"Unique videos            : {ae_video_representation_df[VIDEO_ID_COLUMN].nunique():,}")
print(f"Segment representations  : {len(ae_segment_representation_df):,}")
print(f"Embedding dimensions     : {len(video_embedding_columns):,}")

print("\nRepresentation Statistics")
print("-" * 60)
print(f"Minimum segment count    : {int(ae_video_representation_df['segment_count'].min())}")
print(f"Maximum segment count    : {int(ae_video_representation_df['segment_count'].max())}")
print(f"Mean segment count       : {ae_video_representation_df['segment_count'].mean():.2f}")

print("\nGenerated Outputs")
print("-" * 60)
print(f"Video representations    : {AUTOENCODER_VIDEO_REPRESENTATIONS_CSV.name}")
print(f"Segment representations  : {AUTOENCODER_SEGMENT_REPRESENTATIONS_CSV.name}")
print(f"Representation summary   : {autoencoder_summary_csv.name}")
print(f"Representation directory : {AUTOENCODER_REPRESENTATIONS_DRIVE_DIR}")

print("\nGenerated Artifacts")
print("-" * 60)
print("- Autoencoder video representations (train, validation, test)")
print("- Autoencoder segment representations (train, validation, test)")
print("- Autoencoder representation summary")
print("- Schema validation report")
print("- Sample representation records")

print("\nCompatibility")
print("-" * 60)
print("Representation artifacts follow the shared Notebook 07 schema.")
print("Video embeddings use standardized embedding_### columns.")
print("Notebook 07 will select the required training and validation")
print("representations, merge them with the corresponding NExT-QA")
print("annotations and shared CLIP text embeddings, and train/evaluate")
print("the Fusion MLP classifier.")

